# 1. ARVO Router 학습

ARVO train split만 사용해 Expert Router를 학습하고 `models/router-arvo.pkl`로 저장합니다. OpenRouter API는 호출하지 않습니다.

In [1]:
import json
from collections import Counter
from pathlib import Path
from pprint import pprint
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
DATA_DIR = ROOT / 'data' / 'arvo'
MODEL_PATH = ROOT / 'models' / 'router-arvo.pkl'

from llm_security.config import AppConfig
from llm_security.datasets import load_router_samples_jsonl
from llm_security.router import LearnedRouter

## 학습 데이터 확인

In [2]:
manifest = json.loads((DATA_DIR / 'manifest.json').read_text(encoding='utf-8'))
train_samples = load_router_samples_jsonl(DATA_DIR / 'router_train.jsonl')
family_counts = Counter(
    family.value for sample in train_samples for family in sample.labels
)
print('train projects:', manifest['splits']['train']['project_count'])
print('train samples:', len(train_samples))
print('family distribution:')
pprint(dict(sorted(family_counts.items())))

train projects: 165
train samples: 2356
family distribution:
{'control_state_error': 821, 'lifetime_resource': 177, 'memory_bounds': 1358}


## Router 학습 및 저장

In [3]:
config = AppConfig.from_env(ROOT / '.env')
router = LearnedRouter(
    threshold=config.router.threshold,
    max_experts=config.router.max_experts,
    seed=config.runtime.seed,
).fit(train_samples)
router.save(MODEL_PATH)
print('training complete')
print('saved model:', MODEL_PATH)
print('trained families:', list(router.labeler.classes_))

training complete
saved model: C:\Users\junhyun111\Desktop\llm-security\models\router-arvo.pkl
trained families: ['control_state_error', 'lifetime_resource', 'memory_bounds']
